In [1]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1) paper map 로드
paper_map = pd.read_csv("paper_id_map.csv")
paper_map["paper_id"] = paper_map["paper_id"].astype(str)

# 2) 논문 임베딩 로드
paper_embeddings = np.load("paper_embeddings.npy")

# 3) FAISS index 로드
index = faiss.read_index("papers_faiss.index")

# 4) SPECTER 모델 로드
model = SentenceTransformer("sentence-transformers/allenai-specter")

print("paper_map shape:", paper_map.shape)
print("paper_embeddings shape:", paper_embeddings.shape)
print("faiss ntotal:", index.ntotal)

/Users/jiwon/Library/Python/3.12/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 46160.08it/s]


paper_map shape: (53970, 5)
paper_embeddings shape: (53970, 768)
faiss ntotal: 53970


### title + abstract 합치기

In [2]:
def build_text(title, abstract):
    title = "" if pd.isna(title) else str(title).strip()
    abstract = "" if pd.isna(abstract) else str(abstract).strip()
    return (title + " " + abstract).strip()

### query embedding 생성

In [3]:
def encode_text(text):
    vec = model.encode(
        [text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

### 검색 결과를 paper_id_map 기준으로 매핑

In [4]:
def search_index(query_vec, top_k=10, exclude_paper_id=None):
    scores, indices = index.search(query_vec, top_k + 5)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        paper_id = str(paper_map.iloc[idx]["paper_id"])

        if exclude_paper_id is not None and paper_id == str(exclude_paper_id):
            continue

        results.append({
            "row_idx": int(paper_map.iloc[idx]["row_idx"]),
            "paper_id": paper_id,
            "title": paper_map.iloc[idx]["title"],
            "update_date": paper_map.iloc[idx]["update_date"],
            "categories": paper_map.iloc[idx]["categories"],
            "score": float(score)
        })

        if len(results) == top_k:
            break

    return pd.DataFrame(results)

# retrieval 함수

### 1. seed paper 기반

In [5]:
def retrieve_by_seed_paper(seed_title, seed_abstract, top_k=10, exclude_paper_id=None):
    seed_text = build_text(seed_title, seed_abstract)
    seed_vec = encode_text(seed_text)
    return search_index(seed_vec, top_k=top_k, exclude_paper_id=exclude_paper_id)

### 2. keyword 기반

In [6]:
def retrieve_by_query(query_text, top_k=10):
    query_vec = encode_text(query_text)
    return search_index(query_vec, top_k=top_k)

### 3. seed + keyword

In [7]:
def retrieve_by_seed_and_query(
    seed_title,
    seed_abstract,
    query_text,
    alpha=0.7,
    beta=0.3,
    top_k=10,
    exclude_paper_id=None
):
    seed_text = build_text(seed_title, seed_abstract)

    seed_vec = encode_text(seed_text)
    query_vec = encode_text(query_text)

    final_query_vec = alpha * query_vec + beta * seed_vec
    final_query_vec = final_query_vec / (np.linalg.norm(final_query_vec, axis=1, keepdims=True) + 1e-8)
    final_query_vec = final_query_vec.astype("float32")

    return search_index(final_query_vec, top_k=top_k, exclude_paper_id=exclude_paper_id)

---

## case 1. seed paper only

In [8]:
seed_title = "RETHINKING ATTENTION WITH PERFORMERS"
seed_abstract = "We introduce Performers, Transformer architectures which can estimate regular (softmax) full-rank-attention Transformers with provable accuracy, but using only linear (as opposed to quadratic) space and time complexity, without relying on any priors such as sparsity or low-rankness. To approximate softmax attention-kernels, Performers use a novel Fast Attention Via positive Orthogonal Random features approach (FAVOR+), which may be of independent interest for scalable kernel methods. FAVOR+ can be also used to efficiently model kernelizable attention mechanisms beyond softmax. This representational power is crucial to accurately compare softmax with other kernels for the first time on large-scale tasks, beyond the reach of regular Transformers, and investigate optimal attention-kernels. Performers are linear architectures fully compatible with regular Transformers and with strong theoretical guarantees: unbiased or nearly-unbiased estimation of the attention matrix, uniform convergence and low estimation variance. We tested Performers on a rich set of tasks stretching from pixel-prediction through text models to protein sequence modeling. We demonstrate competitive results with other examined efficient sparse and dense attention methods, showcasing effectiveness of the novel attention-learning paradigm leveraged by Performers."

retrieve_by_seed_paper(seed_title, seed_abstract, top_k=5)

,row_idx,paper_id,title,update_date,categories,score
0,32359,2402.07901,FAST: Factorizable Attention for Speeding up T...,2024-02-13,cs.LG cs.AI cs.NA math.NA,0.900647
1,39662,2406.13762,Unveiling the Hidden Structure of Self-Attenti...,2024-11-01,cs.LG cs.AI cs.CL cs.CV stat.ML,0.895728
2,41087,2407.10005,Fine-grained Analysis of In-context Linear Est...,2024-07-16,cs.LG cs.AI cs.CL math.OC,0.895106
3,39665,2406.13781,A Primal-Dual Framework for Transformers and N...,2024-06-21,cs.LG cs.AI cs.CL cs.CV stat.ML,0.889271
4,44637,2409.15097,Efficiently Dispatching Flash Attention For Pa...,2024-09-25,cs.LG cs.AI cs.CL,0.887760


## case 2. keyword only

In [9]:
retrieve_by_query("long sequence attention mechanism", top_k=5)

,row_idx,paper_id,title,update_date,categories,score
0,37641,2405.15731,Understanding the differences in Foundation Mo...,2024-12-10,cs.LG cs.AI cs.SY eess.SY,0.864358
1,39665,2406.13781,A Primal-Dual Framework for Transformers and N...,2024-06-21,cs.LG cs.AI cs.CL cs.CV stat.ML,0.856385
2,17559,2210.16101,A Generic Shared Attention Mechanism for Vario...,2024-04-11,cs.CV cs.AI,0.856295
3,51241,2501.05730,Element-wise Attention Is All You Need,2025-01-13,cs.LG cs.AI,0.855401
4,53921,cs/0606126,May We Have Your Attention: Analysis of a Sele...,2007-05-23,cs.NE cs.AI,0.853955


## case 3. seed + keyword

In [10]:
retrieve_by_seed_and_query(
    seed_title,
    seed_abstract,
    "long sequence attention mechanism",
    alpha=0.7,
    beta=0.3,
    top_k=5
)

,row_idx,paper_id,title,update_date,categories,score
0,39665,2406.13781,A Primal-Dual Framework for Transformers and N...,2024-06-21,cs.LG cs.AI cs.CL cs.CV stat.ML,0.903789
1,32359,2402.07901,FAST: Factorizable Attention for Speeding up T...,2024-02-13,cs.LG cs.AI cs.NA math.NA,0.897128
2,37641,2405.15731,Understanding the differences in Foundation Mo...,2024-12-10,cs.LG cs.AI cs.SY eess.SY,0.897051
3,39662,2406.13762,Unveiling the Hidden Structure of Self-Attenti...,2024-11-01,cs.LG cs.AI cs.CL cs.CV stat.ML,0.892790
4,51241,2501.05730,Element-wise Attention Is All You Need,2025-01-13,cs.LG cs.AI,0.891086
